In [ ]:
import os

print(os.getcwd())

In [ ]:
import numpy as np
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt

# LaTex configuration
plt.rcParams.update({
    "text.usetex": True,              
    "font.family": "serif",            
    "font.serif": ["Computer Modern"], 
    "figure.figsize": [6.0, 4.5],      # 6" 
    "font.size": 12,                   # The same of the thesis
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "lines.linewidth": 1.5,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "xtick.direction": "in",           
    "ytick.direction": "in",
    "xtick.top": True,                 
    "ytick.right": True,
    "savefig.dpi": 300                 
})

# FOR 2 SUBPLOTS (1x2) 
params_1x2 = {
    "figure.figsize": [10.0, 4.8],
    "axes.labelsize": 14,       # Higher to compensate for scaling
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "lines.linewidth": 2.0      # Thicker lines for visibility
}

#  FOR 3 SUBPLOTS (1x3)
params_1x3 = {
    "figure.figsize": [15.0, 5.0],
    "axes.labelsize": 16,       # Even higher because image will be shrunk more
    "axes.titlesize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "lines.linewidth": 2.2
}
# How to use:
#with plt.rc_context(params_1x3):
    #fig, axes = plt.subplots(1, 3)
    
    # ... Your plotting code for axes[0], axes[1], and axes[2] ...


# Data

In [ ]:
today_Myr = 13700

Table A3 from Mollá et al. (2015)
https://ui.adsabs.harvard.edu/abs/2015MNRAS.451.3693M

In [ ]:
MW = {}

MW['R']     = np.array([         0,      1,     2,     3,     4,     5,     6,    7,    8,    9,   10,   11,   12,   13,   14,    15,   16,     17,     18,     19,     20])

MW['stars'] = 10**np.array([np.nan, np.nan, np.nan, 2.43,  2.50,  2.40,  2.25, 2.09, 1.95, 1.79, 1.69, 1.51, 1.38, 1.25, 1.09,  0.94, 0.80, np.nan, np.nan, np.nan, np.nan])
MW['HI']    =     np.array([  9.41,   3.97,   2.37, 2.39,  3.86,  5.06,  5.04, 5.44, 5.69, 7.69, 6.52, 6.16, 5.63, 4.83, 3.65,  2.96, 2.42,   2.15,   1.61,   1.18,    1.1])
MW['H2']    =     np.array([  0.30,   3.82,   5.18, 3.48,  5.69,  8.28,  8.47, 4.59, 3.15, 2.44, 1.96, 1.24, 0.99, 0.57, 0.82,  1.09,  .20,    .13,    .08,    .03, np.nan])
MW['gas']   = MW['HI'] + MW['H2']
MW['SFR']   = 10**np.array([  -.37,   .603,   .706, .983, 1.163, 1.185, 1.181, .963, .723, .594,  .51, .403, .006, .183, -.26, -.132, -.52,   -.68,   -.89,  -1.37,  -1.37])
MW['OH']    =     np.array([  9.02,   8.86,   8.74, 8.62,  8.82,  8.83,  8.77, 8.69, 8.56, 8.60, 8.45, 8.41, 8.44, 8.44, 8.42,  8.14, 8.14,   8.19,   7.96, np.nan, np.nan])
MW['total'] = MW['gas'] + MW['stars']

MW['HI_err'] = np.array([1.00, 1.88, 2.04, 2.12, 2.35, 2.14, 2.06, 1.58, 2.38, 2.13, 2.18, 2.06, 2.17, 2.86, 2.80, 2.69, 2.19, 2.17, 1.77, 1.66, 1.60])
MW['H2_err'] = np.array([0.50, 4.90, 5.39, 2.24, 3.35, 2.87, 1.67, 1.72, 1.42, 0.80, 1.18, 0.80, 0.75, 0.71, 0.94, 1.84, 0.07, 0.05, 0.03, 0.01, np.nan])
MW['gas_err'] = np.sqrt(MW['HI_err']**2 + MW['H2_err']**2)

MW['stars_err_log'] = np.array([np.nan, np.nan, np.nan, 0.01, 0.13, 0.07, 0.07, 0.08, 0.09, 0.10, 0.14, 0.03, 0.01, 0.03, 0.01, 0.01, 0.01, np.nan, np.nan, np.nan, np.nan])

MW['SFR_err_log'] = np.array([
    0.60, 0.60, 0.60,
    0.59, 0.35, 0.24, 0.27, 0.26,
    0.29, 0.25, 0.43, 0.39, 0.65,
    0.48, 0.57, 0.59, 0.15, 0.18,
    0.15, 0.15, 0.15
])

MW['SFR_err'] = np.log(10) * MW['SFR'] * MW['SFR_err_log']

MW['tdep'] = np.full_like(MW['gas'], np.nan, dtype=float)
MW['tdep_err'] = np.full_like(MW['gas'], np.nan, dtype=float)

mask_tdep = np.isfinite(MW['gas']) & np.isfinite(MW['SFR']) & np.isfinite(MW['gas_err']) & np.isfinite(MW['SFR_err']) & (MW['SFR'] > 0) & (MW['gas'] > 0)
MW['tdep'][mask_tdep] = MW['gas'][mask_tdep] / MW['SFR'][mask_tdep]
MW['tdep_err'][mask_tdep] = MW['tdep'][mask_tdep] * np.sqrt(
    (MW['gas_err'][mask_tdep] / MW['gas'][mask_tdep])**2 +
    (MW['SFR_err'][mask_tdep] / MW['SFR'][mask_tdep])**2
)

print(MW['gas'][8])
print(MW['stars'][8])

# Cylindrical layers model


## Intermediate verification

In [ ]:
# Test parameters
Sigma_gas   = MW['gas'][8]   * u.Msun / u.pc**2
Sigma_stars = MW['stars'][8] * u.Msun / u.pc**2
sigmas = np.linspace(5, 50, 100) * u.km / u.s

# Gravity value for plot title
g_val = (2 * np.pi * G * (Sigma_stars + Sigma_gas)).to(u.pc / u.Myr**2).value

def verify_physics(Sigma_g, Sigma_s, sigma):
    # Gravity (pc/Myr^2)
    g    = (2 * np.pi * G * (Sigma_g + Sigma_s)).to(u.pc / u.Myr**2)
    # Structure (z0 in pc, rho0 in Msun/pc^3)
    z0   = (sigma**2 / g).to(u.pc)
    rho0 = (Sigma_g / (2 * z0)).to(u.Msun / u.pc**3)
    return z0.value, rho0.value

# Run verification  ← argumentos corregidos, zip con 2 valores
results = [verify_physics(Sigma_gas, Sigma_stars, s) for s in sigmas]
z0_vals, rho0_vals = zip(*results)

# Verification plots
with plt.rc_context(params_1x2):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))

    ax[0].plot(sigmas.value, z0_vals, linewidth=2, color='#2E86AB')
    ax[0].set_title(f"Disk Expansion: $z_0$ vs $\\sigma$ ($g = {g_val:.3f}$ pc/Myr$^2$)",
                    fontweight='bold')
    ax[0].set_ylabel("Scale height $z_0$ [pc]")
    ax[0].set_xlabel("Velocity dispersion $\\sigma$ [km/s]")
    ax[0].grid(True, alpha=0.3, linestyle='--')

    ax[1].plot(sigmas.value, rho0_vals, linewidth=2, color='#A23B72')
    ax[1].set_title(f"Central Gas Density: $\\rho_0$ vs $\\sigma$ ($g = {g_val:.3f}$ pc/Myr$^2$)",
                    fontweight='bold')
    ax[1].set_ylabel(r"Central gas density $\rho_0$ [M$_\odot$/pc$^3$]")
    ax[1].set_xlabel("Velocity dispersion $\\sigma$ [km/s]")
    ax[1].grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig('verification_physics.pdf', bbox_inches='tight')
    plt.show()

# Main block

## Using $t_{cooling}$ from Sedov-Taylor phase and constant SFE

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self):
        # Unit configuration (Msun, pc, Myr)
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value
        self.Area = np.pi * (8000) ** 2

        # Physical constants (Equation 2.20 Mario's Thesis)
        self.mH_sim = (1.67e-27 * u.kg).to(u.Msun).value
        self.kB_sim = (
            (1.38e-23 * u.J / u.K).to(u.Msun * u.pc**2 / (u.Myr**2 * u.K)).value
        )

        # Article parameters
        self.a_idx = -0.9  # Index 'a' of cooling curve
        self.Ta = 1e5  # Reference temperature (K)
        self.Lambda_a = 1e-22  # erg * cm^3 / s
        self.n0 = 1.0  # Ambient number density (cm^-3)

        # Conversion factors for t_cool calculation (CGS)
        self.mH_cgs = 1.67e-24  # g
        self.kB_cgs = 1.38e-16  # erg/K
        self.n0_cgs = self.n0  # cm^-3
        self.Lambda_a_cgs = self.Lambda_a  # erg * cm^3 / s

        # E0 (Supernova energy)
        self.E0_cgs = 1e51  # erg

        v0_phys = 220 * u.km / u.s
        self.v0_sim_sq = v0_phys.to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value

        # Stability parameters
        self.eps = 1e-4
        self.M_floor = 1e6

        self.eta_sn = 0.01
        self.sfe = 0.02  # Constant SFE
        self.infall_rate = 1.0e6
        self.tau_inf = 1500.0  # Tau for exponential infall

        self.chi = 2  # Chi parameter 

    def get_physics(self, Mg, Ms, E):
        Mg_eff = Mg + self.eps
        E_eff = E + self.eps

        sigma_sq = 0.4 * (E_eff / Mg_eff)

        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = sigma_sq / (g + 1e-15)

        rho0 = Mg_eff / (2 * self.Area * z0 + 1e-15)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        # T_COOL calculation (Equation 2.20)
        a = self.a_idx

        # Constant term 1
        term_const = (81 * (1 - a) * self.mH_cgs) / (
            1600 * self.n0_cgs * self.Lambda_a_cgs
        )

        # Temperature term (Term 2)
        term_temp = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Energy E0 term (Term 3) - Using constant E0 1e51 erg and n0 without mu
        term_E = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Solve for t_cool in seconds and convert to Myr
        t_cool_s = (term_const * term_temp * (term_E**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13  # from s to Myr

        # Ensure reasonable physical value (in Myr)
        t_cool = np.clip(t_cool, 0.1, 13000.0)

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        # infall = self.infall_rate * np.exp(-t / self.tau_inf) # For exponential infall
        infall = self.infall_rate  # Constant infall
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr

        # E_DOT
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool

        dE = e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool

        return [dMg, dMs, dE]

# Execution
model = GalacticEvolution_Refined()
sigma_init_sim = (15 / 0.9778)
y0 = [1e4, 1e-6, (sigma_init_sim**2 * 1e4) / 0.4]
time_test = 14000

sol = solve_ivp(model.derivatives, [0, time_test], y0, method="LSODA", rtol=1e-8)

# Processing and plots
Mg_s, Ms_s, E_s = sol.y
physics_results = [model.get_physics(Mg_s[i], Ms_s[i], E_s[i]) for i in range(len(sol.t))]
sigma_sq_res, _, _, _ = zip(*physics_results)
sigma_res = np.sqrt(np.array(sigma_sq_res)) * 0.9778 

# Plot 
with plt.rc_context(params_1x2):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    
    ax[0].plot(sol.t, Mg_s / 1e9, label=r"Gas",linewidth = 2, color='#2E86AB')
    ax[0].plot(sol.t, Ms_s / 1e9, label=r"Stars", linewidth=2,color='#F18F01')
    ax[0].set_title(r"Mass Evolution")
    ax[0].set_xlabel(r"Time [Myr]")
    ax[0].set_ylabel(r"Mass [$10^9$ M$_\odot$]")
    ax[0].legend(frameon=True)
    ax[0].grid(True, alpha=0.3, linestyle='--')
    
    ax[1].plot(sol.t, sigma_res, color='#06A77D', linewidth=2)
    ax[1].set_title(r"Velocity Dispersion $\sigma$")
    ax[1].set_xlabel(r"Time [Myr]")
    ax[1].set_ylabel(r"$\sigma$ [km s$^{-1}$]")
    ax[1].grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig('mass_evolution_constant_infall.pdf', bbox_inches='tight')
    plt.show()

## DIAGNOSTIC CALCULATIONS

In [ ]:
Mg_v, Ms_v, E_v = sol.y

# Epicyclic frequency (kappa)
v_c_sim = (220 * u.km / u.s).to_value(u.pc / u.Myr)
kappa = np.sqrt(2) * v_c_sim / 8000.0  

# Toomre stability parameter Q
sigma_v = np.sqrt(0.4 * np.maximum(E_v, 0) / np.maximum(Mg_v, 1e-6))
Sigma_gas = Mg_v / model.Area 
Q_toomre = (sigma_v * kappa) / (np.pi * model.G_sim * Sigma_gas + 1e-15)

# Virial Ratio (K / |U|)
g_v = 2 * np.pi * model.G_sim * (Mg_v + Ms_v) / model.Area
z0_v = sigma_v**2 / (g_v + 1e-15)

# Geometric stability (z0 / R)
thickness_ratio = z0_v / 8000.0

# VERIFICATION PLOTS
with plt.rc_context(params_1x2):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot Toomre Q
    axes[0].plot(sol.t, Q_toomre, linewidth=2.2 , color='#9D4EDD')
    axes[0].axhline(1, color='#C1121F', linestyle='--', label=r"Instability Limit")
    axes[0].set_title(r"Toomre $Q$ Parameter")
    axes[0].set_ylabel(r"$Q$")
    axes[0].set_xlabel(r"Time [Myr]")
    # axes[0].set_xscale("log")
    # Limits for the inestability zone
    # axes[0].set_ylim(0, 10) 
    axes[0].set_yscale("log")
    axes[0].legend(frameon=True)
    axes[0].grid(True, alpha=0.3, linestyle='--')
    
    # Plot Disk Structure (z0 and z0/R)
    line1 = axes[1].plot(sol.t, z0_v, linewidth=2, color='#780000', label=r"$z_0$")
    axes[1].set_ylabel(r"Scale height $z_0$ [pc]")
    axes[1].set_ylim(0, 1000)
    axes[1].grid(True, alpha=0.3, linestyle='--')
    
    ax_twin = axes[1].twinx()
    line2 = ax_twin.plot(sol.t, thickness_ratio, linewidth=2, color='#333333', linestyle='--', label=r"$z_0/R$")
    ax_twin.set_ylabel(r"Ratio $z_0/R$")
    ax_twin.set_ylim(0, 0.5)
    
    lns = line1 + line2
    labs = [l.get_label() for l in lns]
    axes[1].legend(lns, labs, loc='upper right', frameon=True)
    axes[1].set_title(r"Disk Structure")
    axes[1].set_xlabel(r"Time [Myr]")
    
    plt.tight_layout()
    plt.savefig('diagnostic_calculations.pdf', bbox_inches='tight')
    plt.show()

# Exponential infall

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self, radius_pc=8000, tau_inf=12000.0):
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value

        # GEOMETRY: Local ring of 1kpc width
        self.dr = 1000
        self.Area = 2 * np.pi * radius_pc * self.dr

        # RADIAL INFALL (Surface density profile)
        self.t_univ = 13700.0
        Rd = 3000  # Disk scale radius (typical MW)

        sigma_0_inf = 2000.0
        sigma_r_total = sigma_0_inf * np.exp(-radius_pc / Rd)

        self.M_total_anillo = sigma_r_total * self.Area

        self.tau_inf = tau_inf
        self.I0 = self.M_total_anillo / self.tau_inf

        # Physical Parameters
        self.v0_sim_sq = (220 * u.km / u.s).to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value
        self.sfe = 0.004
        self.eta_sn = 0.01
        self.eps = 1e-4

        # Constants for T_COOL
        self.a_idx = -0.9  # Adjusted to article value
        self.chi = 2  # try 0.1
        self.Ta = 1e5  # K
        self.Lambda_a_cgs = 1e-22
        self.n0_cgs = 1.0
        self.mH_cgs = 1.67e-24
        self.kB_cgs = 1.38e-16
        self.E0_cgs = 1e51  # Constant SN energy

    def get_physics(self, Mg, Ms, E):
        Mg_eff, E_eff = Mg + self.eps, E + self.eps
        sigma_sq = 0.4 * (E_eff / Mg_eff)
        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = sigma_sq / (g + 1e-15)  # Using g directly to avoid singularities
        rho0 = Mg_eff / (2 * self.Area * z0 + 1e-15)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        # T_COOL calculation (Equation 2.20)
        a = self.a_idx

        # Constant term 1
        term_const = (81 * (1 - a) * self.mH_cgs) / (
            1600 * self.n0_cgs * self.Lambda_a_cgs
        )

        # Temperature term (Term 2)
        term_temp = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Energy E0 term (Term 3)
        term_E = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Solve for t_cool in seconds and convert to Myr
        t_cool_s = (term_const * term_temp * (term_E**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13

        # Safety clip for integrator stability
        t_cool = np.clip(t_cool, 0.1, 13000.0)

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        # Exponential Infall
        infall = self.I0 * np.exp(-t / self.tau_inf)
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool
        dE = e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool
        return [dMg, dMs, dE]


# Execution (8 kpc)
model_8kpc = GalacticEvolution_Refined(radius_pc=8000)
sigma_init = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0 = [1e4, 1e-6, 2.5 * 1e4 * (sigma_init**2)]

sol = solve_ivp(model_8kpc.derivatives, [0, today_Myr], y0, method="LSODA", rtol=1e-6)

t_v = sol.t
Mg_v = sol.y[0]
Ms_v = sol.y[1]
E_v = sol.y[2]
sigma_v = np.sqrt(0.4 * E_v / (Mg_v + 1e-6))

## Calculations for verification of specific values (ideally calculate by hand and verify)

In [ ]:
# Test values
M_g_test, M_s_test, E_test = 2e9, 5e8, 1e12  # Arbitrary values for testing

# Use 8kpc model defined above
s_sq, tff, z_zero, t_c = model_8kpc.get_physics(M_g_test, M_s_test, E_test)

print("--- FORMULA VERIFICATION ---")
print(f"Gas Mass: {M_g_test:.2e} Msun")
print(f"Stellar Mass: {M_s_test:.2e} Msun")
print(f"Sigma_gas: {M_g_test / model_8kpc.Area:.4e} Msun/pc^2")
print(f"z0 (Scale height): {z_zero:.2f} pc")
print(
    f"rho0 (Central density): {M_g_test / (2 * model_8kpc.Area * z_zero):.4e} Msun/pc^3"
)
print(f"t_ff (Free-fall time): {tff:.2f} Myr")

# Calculation for different radii

In [ ]:
import numpy as np

radios_kpc = np.arange(0.001, 23,1)

sigma_star = []
sigma_gas = []
sigma_sfr = []

print(f"{'Radius [kpc]':<15} | {'log10(Sigma_star)':<20} | {'Sigma_SFR':<20}")
print("-" * 60)

for r in radios_kpc:
    # Create a model for each radius (changes Area and local Infall if desired)
    m = GalacticEvolution_Refined(radius_pc=r * 1000)

    # Assume local Infall decays radially as in a disk (optional)
    # For simplicity here we use global I0, but Mollá uses Sigma_inf(R)
    res = solve_ivp(m.derivatives, [0, today_Myr], y0, method="RK45")

    # Final values (t = today_Myr)
    Mg_f, Ms_f, E_f = res.y[:, -1]
    _, t_ff_f, _, _ = m.get_physics(Mg_f, Ms_f, E_f)

    sfr_f = m.sfe * Mg_f / t_ff_f
    area_kpc2 = m.Area / 1e6  # convert pc^2 to kpc^2

    # Surface densities
    s_star = Ms_f / area_kpc2
    s_sfr = (sfr_f / 1e6) / area_kpc2  # Msun / yr / kpc^2

    l_s_star = np.log10(s_star + 1e-10)

    sigma_star.append(s_star)
    sigma_gas.append(Mg_f / area_kpc2)
    sigma_sfr.append(s_sfr)

    print(f"{r:<15.3f} | {l_s_star:<20.4f} | {s_sfr:<20.4f}")


## Radial profiles

In [ ]:
# Radial Stellar Profile Plot
fig, ax = plt.subplots(figsize=(6, 4.5))

# Model data - Using global rcParams for sizing
ax.plot(radios_kpc, sigma_star, "-", color='k', label="Chemical Evolution Model", zorder=3)
ax.scatter(radios_kpc, sigma_star, color='k', s=40, zorder=4)

mask = np.isfinite(MW['R']) & np.isfinite(MW['stars']) & np.isfinite(MW['stars_err_log'])

stars_obs = MW['stars'][mask] * 1e6
stars_log = np.log10(MW['stars'][mask])

yerr_lower = stars_obs - (10**(stars_log - MW['stars_err_log'][mask])) * 1e6
yerr_upper = (10**(stars_log + MW['stars_err_log'][mask])) * 1e6 - stars_obs

# Mollá et al. (2015) Observation data
ax.errorbar(
    MW['R'][mask],
    stars_obs,
    yerr=[yerr_lower, yerr_upper],
    fmt='none',
    ecolor='0.4',
    elinewidth=1.5,
    capsize=3.5,
    capthick=1.2,
    alpha=0.95,
    zorder=2
)

ax.scatter(
    MW['R'][mask],
    stars_obs,
    s=28,
    facecolor='#E63946',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.95,
    label="Mollá et al. (2015)",
    zorder=3
)

# Axis formatting
ax.set_yscale("log")
ax.set_xlabel(r"Radius $R$ [kpc]")
ax.set_ylabel(r"Stellar Surface Density $\Sigma_\star$ [M$_\odot$ kpc$^{-2}$]")
ax.set_title(f"Final Radial Profile Comparison ($t = {today_Myr/1000:g}$ Gyr)", pad=15, fontweight='bold')

# Limits and Grid
ax.set_xlim(0, 22)
ax.grid(True, which="both", alpha=0.2, linestyle='--')

# Legend
ax.legend(frameon=True, loc='upper right', fancybox=True)

# Save as PDF for LaTeX integration
plt.tight_layout()
plt.savefig('radial_profile_stars.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Radial Gas Profile Plot
fig, ax = plt.subplots(figsize=(6, 4.5))

# Model data - Gas Density 
ax.plot(radios_kpc, sigma_gas, "-", color='k', label="Chemical Evolution Model", zorder=3)
ax.scatter(radios_kpc, sigma_gas, color='k', s=40, marker='s', zorder=4)

mask = np.isfinite(MW['R']) & np.isfinite(MW['gas']) & np.isfinite(MW['gas_err'])

gas_obs = MW['gas'][mask] * 1e6

gas_err_raw = MW['gas_err'][mask] * 1e6
gas_err = np.minimum(gas_err_raw, 0.35 * gas_obs)

# Mollá et al. (2015) Observation data for Gas
ax.errorbar(
    MW['R'][mask],
    gas_obs,
    yerr=gas_err,
    fmt='none',
    ecolor='0.4',
    elinewidth=1.5,
    capsize=3.5,
    capthick=1.2,
    alpha=0.95,
    zorder=2
)

ax.scatter(
    MW['R'][mask],
    gas_obs,
    s=32,
    marker='s',
    facecolor='#457B9D',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.95,
    label="Mollá et al. (2015)",
    zorder=3
)

# Axis formatting
ax.set_yscale("log")
ax.set_xlabel(r"Radius $R$ [kpc]")
ax.set_ylabel(r"Gas Surface Density $\Sigma_{gas}$ [M$_\odot$ kpc$^{-2}$]")
ax.set_title(f"Final Gas Radial Profile Comparison ($t = {today_Myr/1000:g}$ Gyr)", 
             pad=15, fontweight='bold')

# Limits and Grid
ax.set_xlim(0, 22)
ax.grid(True, which="both", alpha=0.2, linestyle='--')

# Legend
ax.legend(frameon=True, loc='upper right')

# Final layout and saving to PDF
plt.tight_layout()
plt.savefig('radial_profile_gas.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Radial SFR Profile Plot
fig, ax = plt.subplots(figsize=(6, 4.5))

# Model data - Star Formation Rate (Inheriting global styles)
ax.plot(radios_kpc, sigma_sfr, "-", color='k', label="Chemical Evolution Model", zorder=3)
ax.scatter(radios_kpc, sigma_sfr, color='k', s=45, marker='^', zorder=4)

mask = np.isfinite(MW['R']) & np.isfinite(MW['SFR']) & np.isfinite(MW['SFR_err'])

sfr_obs = MW['SFR'][mask] * 1e6 / 1e9

sfr_err_raw = MW['SFR_err'][mask] * 1e6 / 1e9
sfr_err = np.minimum(sfr_err_raw, 0.35 * sfr_obs)

# Mollá et al. (2015) data - Normalized to consistent units
ax.errorbar(
    MW['R'][mask],
    sfr_obs,
    yerr=sfr_err,
    fmt='none',
    ecolor='0.4',
    elinewidth=1.5,
    capsize=3.5,
    capthick=1.2,
    alpha=0.95,
    zorder=2
)

ax.scatter(
    MW['R'][mask],
    sfr_obs,
    s=34,
    marker='^',
    facecolor='#F4A261',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.95,
    label="Mollá et al. (2015)",
    zorder=3
)

# Axes and scaling
ax.set_yscale("log")
ax.set_xlabel(r"Radius $R$ [kpc]")
ax.set_ylabel(r"Star Formation Rate $\Sigma_{SFR}$ [M$_\odot$ yr$^{-1}$ kpc$^{-2}$]")
ax.set_title(f"Final SFR Radial Profile Comparison ($t = {today_Myr/1000:g}$ Gyr)", 
             pad=15, fontweight='bold')

# Limits and Grid
ax.set_xlim(0, 22)
ax.grid(True, which="both", alpha=0.2, linestyle='--')

# Legend
ax.legend(frameon=True, loc='upper right')

# Final layout and saving to PDF
plt.tight_layout()
plt.savefig('radial_profile_sfr.pdf', bbox_inches='tight')
plt.show()

In [ ]:
#  Gas Depletion Time Radial Profile Plot
fig, ax = plt.subplots()

# Model data - Gas depletion time (t_dep = Sigma_gas / Sigma_SFR)
# Units: (Msun/pc^2) / (Msun/yr/pc^2) * 1e-9 = Gyr
t_dep_model = np.array(sigma_gas) / np.array(sigma_sfr) * 1e-9
ax.plot(radios_kpc, t_dep_model, "-", color='k', label="Chemical Evolution Model", zorder=3)
ax.scatter(radios_kpc, t_dep_model, color='k', s=40, marker='D', zorder=4)

mask = np.isfinite(MW['R']) & np.isfinite(MW['tdep']) & np.isfinite(MW['tdep_err'])

# Mollá et al. (2015) data
# Units converted to Gyr
t_dep_molla = MW['tdep'][mask]

tdep_err_raw = MW['tdep_err'][mask]
tdep_err = np.minimum(tdep_err_raw, 0.35 * t_dep_molla)

ax.errorbar(
    MW['R'][mask],
    t_dep_molla,
    yerr=tdep_err,
    fmt='none',
    ecolor='0.25',
    elinewidth=1.5,
    capsize=3.5,
    capthick=1.2,
    alpha=0.95,
    zorder=2
)

ax.scatter(
    MW['R'][mask],
    t_dep_molla,
    s=34,
    marker='D',
    facecolor='#06A77D',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.95,
    label="Mollá et al. (2015)",
    zorder=3
)

# Axes and scaling
ax.set_yscale('log')
ax.set_xlabel(r"Radius $R$ [kpc]")
ax.set_ylabel(r"Gas Depletion Time $\tau_{dep}$ [Gyr]")
ax.set_title(f"Final Depletion Time Profile Comparison ($t = {today_Myr/1000:g}$ Gyr)", 
             fontweight='bold', pad=15)

# Limits and Grid - Setting xlim from 0 to 22 for visual centralization
ax.set_xlim(0, 22)
ax.grid(True, which="both", alpha=0.2, linestyle='--')

# Legend
ax.legend(frameon=True, loc='upper left')

# Final layout and saving as PDF for vector quality
plt.tight_layout()
plt.savefig('radial_profile_depletion_time.pdf', bbox_inches='tight')
plt.show()

## Negligible initial mass and $\tau $ variable. Inside - out model

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt


class GalacticEvolution_Refined:
    def __init__(self, radius_pc=8000, tau_inf=2000.0):
        self.G_sim = G.to(u.pc**3 / (u.Msun * u.Myr**2)).value
        self.radius_pc = radius_pc
        self.dr = 1000
        self.Area = 2 * np.pi * radius_pc * self.dr

        # Mollá Infall Profile
        Rd = 3500.0
        sigma_0_inf = 1500.0
        sigma_r_total = sigma_0_inf * np.exp(-radius_pc / Rd)

        self.M_total_anillo = sigma_r_total * self.Area
        self.tau_inf = tau_inf
        self.I0 = self.M_total_anillo / self.tau_inf

        self.v0_sim_sq = (220 * u.km / u.s).to(u.pc / u.Myr).value ** 2
        self.E_sn_sim = (1e51 * u.erg).to(u.Msun * u.pc**2 / u.Myr**2).value
        self.sfe = 0.02
        self.eta_sn = 0.01
        self.eps = 1e-4

        # Physical constants for T_COOL (CGS)
        self.a_idx = -0.9  # Article value
        self.chi = 2
        self.Ta = 1e5  # K
        self.Lambda_a_cgs = 1e-22
        self.n0_cgs = 1.0
        self.mH_cgs = 1.67e-24
        self.kB_cgs = 1.38e-16
        self.E0_cgs = 1e51  # Constant SN energy

    def get_physics(self, Mg, Ms, E):
        Mg_eff = Mg + self.eps
        sigma_sq = 0.4 * (E / Mg_eff)

        g = 2 * np.pi * self.G_sim * (Mg_eff + Ms) / self.Area
        z0 = (sigma_sq / (g + 1e-15)) + 1.0  # The +1 avoids initial divisions by zero
        rho0 = Mg_eff / (2 * self.Area * z0)
        t_ff = np.sqrt((3 * np.pi) / (32 * self.G_sim * (rho0 + 1e-15)))

        a = self.a_idx

        # Term 1 (Cooling constants)
        term1 = (81 * (1 - a) * self.mH_cgs) / (1600 * self.n0_cgs * self.Lambda_a_cgs)

        # Term 2 (Reference temperature)
        term2 = ((9 * self.mH_cgs) / (80 * self.kB_cgs * self.Ta)) ** (-a)

        # Term 3 (Injected SN energy)
        term3 = (16 * self.chi * self.E0_cgs) / (375 * self.n0_cgs * self.mH_cgs)

        expo_E = (2 / 5) * (1 - a)
        expo_final = (1 / 5) * (11 - 6 * a)

        # Calculate t_cool in seconds and convert to Myr
        t_cool_s = (term1 * term2 * (term3**expo_E)) ** (1 / expo_final)
        t_cool = t_cool_s / 3.154e13

        return sigma_sq, t_ff, z0, t_cool

    def derivatives(self, t, y):
        Mg, Ms, E = y
        sigma_sq, t_ff, z0, t_cool = self.get_physics(Mg, Ms, E)

        infall = self.I0 * np.exp(-t / self.tau_inf)
        sfr = self.sfe * max(Mg, 0) / t_ff

        dMg = infall - sfr
        dMs = sfr
        e_dot_in = infall * (0.5 * self.v0_sim_sq + sigma_sq)
        e_dot_sn = sfr * self.eta_sn * self.E_sn_sim
        e_dot_sfr = sfr * (E / (Mg + self.eps))
        e_dot_cool = E / t_cool

        return [dMg, dMs, e_dot_in + e_dot_sn - e_dot_sfr - e_dot_cool]


# Execution 8kpc
model_8kpc = GalacticEvolution_Refined(radius_pc=8000, tau_inf=7000.0)
sigma_init = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0 = [1e6, 0, 2.5 * 1e6 * (sigma_init**2)]
sol = solve_ivp(model_8kpc.derivatives, [0, today_Myr], y0, method="LSODA")

t_v = sol.t
Mg_v = sol.y[0]
Ms_v = sol.y[1]
E_v = sol.y[2]
sigma_v = np.sqrt(0.4 * E_v / (Mg_v + 1e-6))

## Verification

In [ ]:
# Test values
M_g_test, M_s_test, E_test = 3e10, 0, 1e14  # Arbitrary values for testing

# Use 8kpc model defined above
s_sq, tff, z_zero, t_c = model_8kpc.get_physics(M_g_test, M_s_test, E_test)

print("--- FORMULA VERIFICATION ---")
print(f"Gas Mass: {M_g_test:.2e} Msun")
print(f"Stellar Mass: {M_s_test:.2e} Msun")
print(f"Sigma_gas: {M_g_test / model_8kpc.Area:.4e} Msun/pc^2")
print(f"z0 (Scale height): {z_zero:.2f} pc")
print(
    f"rho0 (Central density): {M_g_test / (2 * model_8kpc.Area * z_zero):.4e} Msun/pc^3"
)
print(f"t_ff (Free-fall time): {tff:.2f} Myr")

## Density calculation for different radii

In [ ]:
# Define radius range for the simulation
radios = np.arange(0.001, 23, 1)
log_sigma_star = []
sfr_vals = []

# Inside-Out parameters for a more gradual growth
# We use a linear increase for tau to prevent early gas exhaustion in the outskirts
tau_center = 1000 
tau_gradient = 1500 # Myr per kpc

for r in radios:
    # Linear inside-out law (more robust than pure exponential for wide disks)
    tau_r = tau_center + tau_gradient * r
    
    m = GalacticEvolution_Refined(radius_pc=r * 1000, tau_inf=tau_r)

    # Maintain initial conditions with the required 220 km/s energy
    Mg0_r = 1e6
    E0_r = (Mg0_r * 220**2) / 0.4

    # Integrate the system
    res = solve_ivp(m.derivatives, [0, today_Myr], [Mg0_r, 0, E0_r], method="LSODA")
    Mg_f, Ms_f, E_f = res.y[:, -1]
    
    # Calculate final physics
    _, t_ff_f, _, _ = m.get_physics(Mg_f, Ms_f, E_f)

    # Store Stellar Surface Density (log10 Msun/pc^2)
    log_sigma_star.append(np.log10(Ms_f / m.Area + 1e-10))
    
    # Calculate SFR Surface Density in Msun / pc^2 / Gyr
    # Dividing by area and converting Myr to Gyr
    sfr_density_gyr = ((m.sfe * Mg_f / t_ff_f) / m.Area) * 1000
    sfr_vals.append(sfr_density_gyr)

# Convert results to log scale for plotting
log_psi = np.log10(np.array(sfr_vals) + 1e-12)
mw_sfr_log = np.log10(MW['SFR'])

# Plotting diagnostics
with plt.rc_context(params_1x2):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    
    # Stellar Density Panel
    ax[0].plot(radios, log_sigma_star, "o-", color='#2E86AB', label="Model")

    mask_star = np.isfinite(MW['R']) & np.isfinite(MW['stars']) & np.isfinite(MW['stars_err_log'])

    ax[0].errorbar(
        MW['R'][mask_star],
        np.log10(MW['stars'][mask_star]),
        yerr=MW['stars_err_log'][mask_star],
        fmt='none',
        ecolor='0.4',
        elinewidth=1.4,
        capsize=3,
        capthick=1.1,
        alpha=0.95,
        zorder=2
    )

    ax[0].scatter(
        MW['R'][mask_star],
        np.log10(MW['stars'][mask_star]),
        s=28,
        facecolor='#E63946',
        edgecolor='black',
        linewidth=0.6,
        alpha=0.95,
        label="Mollá et al. (2015)",
        zorder=3
    )

    ax[0].set_title(r"Stellar Surface Density $\log \Sigma_{\star}$")
    ax[0].set_xlabel(r"Radius [kpc]")
    ax[0].set_ylabel(r"$\log \Sigma_{\star}$ [M$_\odot$ pc$^{-2}$]")
    ax[0].legend(frameon=True)
    
    # SFR Density Panel
    ax[1].plot(radios, log_psi, "s-", color='#F77F00', label="Model")

    mask_sfr = np.isfinite(MW['R']) & np.isfinite(mw_sfr_log) & np.isfinite(MW['SFR_err_log'])

    ax[1].errorbar(
        MW['R'][mask_sfr],
        mw_sfr_log[mask_sfr],
        yerr=np.minimum(MW['SFR_err_log'][mask_sfr], 0.35),
        fmt='none',
        ecolor='0.4',
        elinewidth=1.4,
        capsize=3,
        capthick=1.1,
        alpha=0.95,
        zorder=2
    )

    ax[1].scatter(
        MW['R'][mask_sfr],
        mw_sfr_log[mask_sfr],
        s=30,
        facecolor='#E63946',
        edgecolor='black',
        linewidth=0.6,
        alpha=0.95,
        label="Mollá et al. (2015)",
        zorder=3
    )

    ax[1].set_title(r"Star Formation Rate $\log(\Sigma_{SFR})$")
    ax[1].set_xlabel(r"Radius [kpc]")
    ax[1].set_ylabel(r"$\log \Sigma_{SFR}$ [M$_\odot$ pc$^{-2}$ Gyr$^{-1}$]")
    ax[1].legend(frameon=True)
    
    plt.tight_layout()
    plt.savefig('radial_profiles_varying_tau.pdf',bbox_inches='tight')
    plt.show()

In [ ]:
t = sol.t
Mg, Ms, E = sol.y

# Physical calculations
sigma = np.sqrt(0.4 * np.maximum(E, 0) / np.maximum(Mg, 1e-6))
g = 2 * np.pi * model_8kpc.G_sim * (Mg + Ms) / model_8kpc.Area
z0 = sigma**2 / (g + 1e-15)

v_c_sim = (220 * u.km / u.s).to_value(u.pc / u.Myr)
kappa = np.sqrt(2) * v_c_sim / model_8kpc.radius_pc  
Q = (sigma * kappa) / (np.pi * model_8kpc.G_sim * (Mg / model_8kpc.Area) + 1e-15)
thickness_ratio = z0 / model_8kpc.radius_pc

# Mass Evolution, Stellar Density and Velocity Dispersion
with plt.rc_context(params_1x3):
    fig1, axes1 = plt.subplots(1, 3, figsize=(14, 4.5))
    
    axes1[0].plot(t, Mg, label="Gas", color='#2E86AB', linewidth=2.5)
    axes1[0].plot(t, Ms, label="Stars", color='#F18F01', linewidth=2.5)
    axes1[0].set_title("Mass Evolution", fontweight='bold')
    axes1[0].set_xlabel("Time [Myr]")
    axes1[0].set_ylabel("Mass [M$_\odot$]")
    axes1[0].legend(frameon=True, loc='best')
    axes1[0].grid(True, alpha=0.3, linestyle='--')
    
    axes1[1].plot(t, np.log10(Ms / model_8kpc.Area + 1e-10), color='#023047', linewidth=2.5)
    axes1[1].set_title(r"Stellar Density $\log \Sigma_{\star}$", fontweight='bold')
    axes1[1].set_xlabel("Time [Myr]")
    axes1[1].set_ylabel(r"$\log \Sigma_{\star}$ [M$_\odot$/pc$^2$]")
    axes1[1].grid(True, alpha=0.3, linestyle='--')
    
    axes1[2].plot(t, sigma, color='#06A77D', linewidth=2.0)
    axes1[2].set_title(r"Velocity Dispersion $\sigma$", fontweight='bold')
    axes1[2].set_xlabel("Time [Myr]")
    axes1[2].set_ylabel(r"$\sigma$ [pc/Myr]")
    axes1[2].grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig('mass_structure_evolution.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# Toomre Q and Disk Structure (z0 & z0/R)
with plt.rc_context(params_1x2):
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4.5))
    
    axes2[0].plot(t, Q, linewidth=2.2, color='#9D4EDD')
    axes2[0].axhline(1, color='#C1121F', linestyle='--', label=r"Instability Limit")
    axes2[0].set_title(r"Toomre $Q$ Parameter")
    axes2[0].set_ylabel(r"$Q$")
    axes2[0].set_xlabel(r"Time [Myr]")
    # axes2[0].set_xscale("log")
    # axes2[0].set_ylim(0, 10)
    axes2[0].set_yscale("log")
    axes2[0].legend(frameon=True)
    axes2[0].grid(True, alpha=0.3, linestyle='--')
    
    line1 = axes2[1].plot(t, z0, linewidth=2, color='#780000', label=r"$z_0$")
    axes2[1].set_ylabel(r"Scale height $z_0$ [pc]")
    axes2[1].set_ylim(0, 200)
    axes2[1].grid(True, alpha=0.3, linestyle='--')
    
    ax_twin = axes2[1].twinx()
    line2 = ax_twin.plot(t, thickness_ratio, linewidth=2, color='#333333', linestyle='--', label=r"$z_0/R$")
    ax_twin.set_ylabel(r"Ratio $z_0/R$")
    ax_twin.set_ylim(0, 0.2)
    
    lns = line1 + line2
    labs = [l.get_label() for l in lns]
    axes2[1].legend(lns, labs, loc='upper right', frameon=True)
    axes2[1].set_title(r"Disk Structure")
    axes2[1].set_xlabel(r"Time [Myr]")
    
    plt.tight_layout()
    plt.savefig('dynamics_stability.pdf', bbox_inches='tight')
    plt.show()

# Varying $\tau$

In [ ]:
# Block 1: Diagnostic and Plotting Function
import numpy as np
from scipy.integrate import solve_ivp
import astropy.units as u
from astropy.constants import G
import matplotlib.pyplot as plt

def ejecutar_diagnostico(modelo, t_span, y0):
    sol = solve_ivp(modelo.derivatives, t_span, y0, method="LSODA", rtol=1e-6)
    t = sol.t
    Mg, Ms, E = sol.y[0], sol.y[1], sol.y[2]

    # Velocity dispersion and disk height
    sigma_sq = 0.4 * E / (Mg + 1e-6)
    sigma = np.sqrt(sigma_sq)
    g = 2 * np.pi * modelo.G_sim * (Mg + Ms) / modelo.Area
    z0 = sigma_sq / (g + 1e-15)

    # Virial ratio
    K_turb = 1.5 * Mg * sigma_sq
    U_grav = 1.0 * Mg * sigma_sq
    virial_ratio = K_turb / (U_grav + 1e-15)

    # Toomre stability parameter
    v_circ_sim = (220 * u.km / u.s).to(u.pc / u.Myr).value
    kappa = np.sqrt(2) * v_circ_sim / modelo.radius_pc
    Q = (sigma * kappa) / (np.pi * modelo.G_sim * (Mg / modelo.Area) + 1e-15)

    return t, Mg, Ms, sigma, Q, virial_ratio, z0

In [ ]:
# Block 2: Model Setup and Normalization
t_span = [0, today_Myr]
radius_8kpc = 8000
sigma_init_val = (15 * u.km / u.s).to_value(u.pc / u.Myr)
y0_despreciable = [1e4, 0, (1e4 * sigma_init_val**2) / 0.4]

# Target surface density to match Milky Way observations at 13.8 Gyr
sigma_star_obs = MW['stars'][8]   # ~89 Msun/pc^2
sigma_gas_obs  = MW['gas'][8]     # ~8.8 Msun/pc^2

target_sigma_total = sigma_star_obs + sigma_gas_obs

print("Target surface density =", target_sigma_total)

# Initialize models with different timescales
model_fast = GalacticEvolution_Refined(radius_pc=radius_8kpc, tau_inf=1000.0)
model_const = GalacticEvolution_Refined(radius_pc=radius_8kpc, tau_inf=1e9)

# Manually override I0 to ensure final integrated mass is identical for both cases
M_total_deseada_f = target_sigma_total * model_fast.Area
model_fast.I0 = M_total_deseada_f / (model_fast.tau_inf * (1 - np.exp(-today_Myr / model_fast.tau_inf)))

M_total_deseada_c = target_sigma_total * model_const.Area
model_const.I0 = M_total_deseada_c / (model_const.tau_inf * (1 - np.exp(-today_Myr / model_const.tau_inf)))

# Execute simulations
data_fast = ejecutar_diagnostico(model_fast, t_span, y0_despreciable)
data_const = ejecutar_diagnostico(model_const, t_span, y0_despreciable)

In [ ]:
# FIGURE 1: Mass Components, Stellar Density, and Velocity Dispersion
with plt.rc_context(params_1x3):
    
    fig1, axes1 = plt.subplots(1, 3, figsize=(14, 4.5))
    
    # Plotting Fast Infall
    t_f, Mg_f, Ms_f, sigma_f, Q_f, virial_f, z0_f = data_fast
    axes1[0].plot(t_f, Mg_f, color='#1D3557', ls='-', lw=2.5, label=r"Gas (Fast)")
    axes1[0].plot(t_f, Ms_f, color='#E63946', ls='-', lw=2.5, label=r"Stars (Fast)")
    axes1[1].plot(t_f, np.log10(Ms_f / model_fast.Area + 1e-10), color='#1D3557', ls='-', lw=2.5, label=r"Fast")
    axes1[2].plot(t_f, sigma_f, color='#2D6A4F', ls='-', lw=2.5, label="Fast")
    
    # Plotting Constant Infall
    t_c, Mg_c, Ms_c, sigma_c, Q_c, virial_c, z0_c = data_const
    axes1[0].plot(t_c, Mg_c, color='#457B9D', ls='--', lw=2.5, label=r"Gas (Const)")
    axes1[0].plot(t_c, Ms_c, color='#F4A261', ls='--', lw=2.5, label=r"Stars (Const)")
    axes1[1].plot(t_c, np.log10(Ms_c / model_const.Area + 1e-10), color='#457B9D', ls='--', lw=2.5, label=r"Constant")
    axes1[2].plot(t_c, sigma_c, color='#D00000', ls='--', lw=2.5, label="Constant")
    
    # Formatting
    axes1[0].set_yscale("log")
    axes1[0].set_title("Mass Components", fontweight='bold')
    axes1[0].set_ylabel("Mass [M$_\odot$]")
    
    axes1[1].set_title(r"Stellar Surface Density $\log \Sigma_{\star}$", fontweight='bold')
    axes1[1].set_ylabel(r"$\log$ [M$_\odot$ pc$^{-2}$]")
    
    axes1[2].set_title(r"Velocity Dispersion $\sigma$", fontweight='bold')
    axes1[2].set_ylabel(r"$\sigma$ [pc Myr$^{-1}$]")
    
    for ax in axes1:
        ax.set_xlabel("Time [Myr]")
        ax.legend(frameon=True)
        ax.grid(True, alpha=0.2, linestyle='--', which="both")
    
    plt.tight_layout()
    plt.savefig('comparison_tau_mass_evolution.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
with plt.rc_context(params_1x2):
    
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4.5))
    
    mask_f = t_f > 50
    mask_c = t_c > 50
    
    kw_f = dict(color='#780000', ls='-',  lw=2.5, label=r'Fast ($\tau=1000$ Myr)')
    kw_c = dict(color='#333333', ls='--', lw=2.5, label=r'Constant ($\tau=10^9$ Myr)')
    
    # Panel 1: Q
    axes2[0].plot(t_f, Q_f, color='#6A0DAD', label=r'Fast ($\tau=1000$ Myr)')
    axes2[0].plot(t_c, Q_c, color='#FF8C00', label=r'Constant ($\tau=10^9$ Myr)')
    axes2[0].axhline(1.0, color='#C1121F', ls='--', lw=1.8, label='Instability limit')
    axes2[0].set_yscale('log')
    axes2[0].set_title(r'Toomre $Q$ Parameter')
    axes2[0].set_xlabel('Time [Myr]')
    axes2[0].set_ylabel(r'$Q$')
    axes2[0].legend(frameon=True)
    axes2[0].grid(True, alpha=0.2, which='both', linestyle='--')
    
    # Panel 2: z0
    axes2[1].plot(t_f[mask_f], z0_f[mask_f], **kw_f)
    axes2[1].plot(t_c[mask_c], z0_c[mask_c], **kw_c)
    axes2[1].set_yscale('log')
    axes2[1].set_title('Disk Scale Height')
    axes2[1].set_xlabel('Time [Myr]')
    axes2[1].set_ylabel(r'Scale height $z_0$ [pc]')
    axes2[1].legend(frameon=True)
    axes2[1].grid(True, alpha=0.2, which='both', linestyle='--')
    
    
    plt.tight_layout()
    plt.savefig('comparison_tau_dynamics.pdf', bbox_inches='tight')
    plt.show()